# Extraction of file

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Input.xlsx to Input (2).xlsx
Saving analyze (1).py to analyze (1).py


In [4]:
import os
os.listdir()

['.config',
 'Input (1).xlsx',
 'Input (2).xlsx',
 'analyze (1).py',
 'Input.xlsx',
 'sample_data']

In [ ]:
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup

def extract_article_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}   # ✅ added

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    title = soup.find('h1')
    title_text = title.get_text() if title else 'No Title'

    paragraphs = soup.find_all('p')
    article_text = ' '.join([para.get_text() for para in paragraphs if para.get_text().strip()])

    return title_text, article_text

def save_article_text(output_dir, url_id, title, article_text):
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{url_id}.txt")

    with open(file_path, "w", encoding='utf-8') as file:
        file.write(f"{title}\n\n{article_text}")

def main():
    input_df = pd.read_excel('Input.xlsx')   # ✅ same path (Colab me yahi chalega)
    output_dir = 'extracted_articles'

    for index, row in input_df.iterrows():
        url_id = row['URL_ID']
        url = row['URL']

        try:
            title, article_text = extract_article_text(url)
            save_article_text(output_dir, url_id, title, article_text)
            print(f"Done: {url_id}")   # ✅ progress print
        except Exception as e:
            print(f"Failed: {url} | Error: {e}")

if __name__ == "__main__":
    main()

In [6]:
!python extract.py

python3: can't open file '/content/extract.py': [Errno 2] No such file or directory


In [ ]:
import os
os.listdir("extracted_articles")

In [ ]:
from google.colab import files
files.upload()

# ***ANALYZE***

In [112]:
import streamlit as st
import re
import os

# ================== LOAD WORDS ==================
def load_words(file):
    try:
        with open(file, 'r', encoding='latin-1') as f:
            return set([w.strip().lower() for w in f if w.strip()])
    except:
        return set()

# 👇 IMPORTANT: ye global scope me hona chahiye
positive_words = load_words("/content/positive-words.txt")
negative_words = load_words("/content/negative-words.txt")

# ================== FUNCTIONS ==================
def count_syllables(word):
    word = word.lower()
    vowels = "aeiou"
    count = 0

    if word and word[0] in vowels:
        count += 1

    for i in range(1, len(word)):
        if word[i] in vowels and word[i-1] not in vowels:
            count += 1

    if word.endswith("e"):
        count -= 1

    return max(count, 1)

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().split()

def count_pronouns(text):
    return len(re.findall(r'\b(I|we|my|ours|us)\b', text, re.I))

# ================== UI ==================
st.title("📊 Text Analysis App")

text = st.text_area("Enter your text here")

# ================== ANALYZE ==================
if st.button("🚀 Analyze Text"):

    words = clean_text(text)
    word_count = len(words)

    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip() != ""])

    avg_sentence_length = word_count / (sentence_count + 1)
    avg_words_per_sentence = word_count / (sentence_count + 1)

    # ✅ SAFE CALCULATION (error-proof)
    try:
        pos_score = sum(1 for w in words if w in positive_words)
        neg_score = sum(1 for w in words if w in negative_words)
    except:
        pos_score = 0
        neg_score = 0

    polarity = (pos_score - neg_score) / (pos_score + neg_score + 0.000001)
    subjectivity = (pos_score + neg_score) / (word_count + 0.000001)

    syllables = [count_syllables(w) for w in words]
    complex_words = sum(1 for s in syllables if s > 2)

    percentage_complex = complex_words / (word_count + 0.000001)
    fog_index = 0.4 * (avg_sentence_length + percentage_complex)

    avg_syllable = sum(syllables) / (word_count + 0.000001)
    pronouns = count_pronouns(text)
    avg_word_length = sum(len(w) for w in words) / (word_count + 0.000001)

    # ================== OUTPUT ==================
    st.subheader("📊 Full Analysis Results")

    st.write("POSITIVE SCORE:", pos_score)
    st.write("NEGATIVE SCORE:", neg_score)
    st.write("POLARITY SCORE:", round(polarity, 2))
    st.write("SUBJECTIVITY SCORE:", round(subjectivity, 2))

    st.write("AVG SENTENCE LENGTH:", round(avg_sentence_length, 2))
    st.write("PERCENTAGE OF COMPLEX WORDS:", round(percentage_complex, 2))
    st.write("FOG INDEX:", round(fog_index, 2))
    st.write("AVG NUMBER OF WORDS PER SENTENCE:", round(avg_words_per_sentence, 2))

    st.write("COMPLEX WORD COUNT:", complex_words)
    st.write("WORD COUNT:", word_count)
    st.write("SYLLABLE PER WORD:", round(avg_syllable, 2))
    st.write("PERSONAL PRONOUNS:", pronouns)
    st.write("AVG WORD LENGTH:", round(avg_word_length, 2))

2026-03-22 10:35:54.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.276 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.276 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:35:54.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [113]:
st.write("Positive words loaded:", len(positive_words))
st.write("Negative words loaded:", len(negative_words))

2026-03-22 10:36:36.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.526 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 10:36:36.532 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [114]:
import os
print(os.listdir("/content"))

['.config', 'Input (1).xlsx', 'negative-words.txt', 'Input (2).xlsx', 'app.py', 'analyze (1).py', 'MasterDictionary-20260322T052959Z-3-001.zip', 'positive-words.txt', 'StopWords-20260322T053001Z-3-001 (1).zip', 'analyze.py', 'extracted_articles', 'MasterDictionary-20260322T052959Z-3-001 (1).zip', 'Input.xlsx', 'final_output.xlsx', 'StopWords-20260322T053001Z-3-001.zip', 'sample_data']


In [115]:
!python analyze.py

Traceback (most recent call last):
  File "/content/analyze.py", line 2, in <module>
    PASTE_YOUR_ANALYZE_CODE_HERE
NameError: name 'PASTE_YOUR_ANALYZE_CODE_HERE' is not defined


In [116]:
from google.colab import files
files.download("final_output.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ***ML App DEPLOYMENT ON Streamlit***

In [117]:
!pip install streamlit pyngrok

In [118]:
code = """
import streamlit as st
import re
import os

# ================== LOAD WORDS ==================
def load_words(file):
    try:
        with open(file, 'r', encoding='latin-1') as f:
            return set([w.strip().lower() for w in f if w.strip()])
    except:
        return set()

# 👇 IMPORTANT: ye global scope me hona chahiye
positive_words = load_words("positive-words.txt")
negative_words = load_words("negative-words.txt")

# ================== FUNCTIONS ==================
def count_syllables(word):
    word = word.lower()
    vowels = "aeiou"
    count = 0

    if word and word[0] in vowels:
        count += 1

    for i in range(1, len(word)):
        if word[i] in vowels and word[i-1] not in vowels:
            count += 1

    if word.endswith("e"):
        count -= 1

    return max(count, 1)

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().split()

def count_pronouns(text):
    return len(re.findall(r'\b(I|we|my|ours|us)\b', text, re.I))

# ================== UI ==================
st.title("📊 Text Analysis App")

text = st.text_area("Enter your text here")

# ================== ANALYZE ==================
if st.button("🚀 Analyze Text"):

    words = clean_text(text)
    word_count = len(words)

    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip() != ""])

    avg_sentence_length = word_count / (sentence_count + 1)
    avg_words_per_sentence = word_count / (sentence_count + 1)

    # ✅ SAFE CALCULATION (error-proof)
    try:
        pos_score = sum(1 for w in words if w in positive_words)
        neg_score = sum(1 for w in words if w in negative_words)
    except:
        pos_score = 0
        neg_score = 0

    polarity = (pos_score - neg_score) / (pos_score + neg_score + 0.000001)
    subjectivity = (pos_score + neg_score) / (word_count + 0.000001)

    syllables = [count_syllables(w) for w in words]
    complex_words = sum(1 for s in syllables if s > 2)

    percentage_complex = complex_words / (word_count + 0.000001)
    fog_index = 0.4 * (avg_sentence_length + percentage_complex)

    avg_syllable = sum(syllables) / (word_count + 0.000001)
    pronouns = count_pronouns(text)
    avg_word_length = sum(len(w) for w in words) / (word_count + 0.000001)

    # ================== OUTPUT ==================
    st.subheader("📊 Full Analysis Results")

    st.write("POSITIVE SCORE:", pos_score)
    st.write("NEGATIVE SCORE:", neg_score)
    st.write("POLARITY SCORE:", round(polarity, 2))
    st.write("SUBJECTIVITY SCORE:", round(subjectivity, 2))

    st.write("AVG SENTENCE LENGTH:", round(avg_sentence_length, 2))
    st.write("PERCENTAGE OF COMPLEX WORDS:", round(percentage_complex, 2))
    st.write("FOG INDEX:", round(fog_index, 2))
    st.write("AVG NUMBER OF WORDS PER SENTENCE:", round(avg_words_per_sentence, 2))

    st.write("COMPLEX WORD COUNT:", complex_words)
    st.write("WORD COUNT:", word_count)
    st.write("SYLLABLE PER WORD:", round(avg_syllable, 2))
    st.write("PERSONAL PRONOUNS:", pronouns)
    st.write("AVG WORD LENGTH:", round(avg_word_length, 2))
"""

with open("app.py", "w") as f:
    f.write(code)

print("app.py created ✅")

app.py created ✅


<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:37: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_534/1558406057.py:37: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(r'[^a-zA-Z\s]', '', text)


In [119]:
!pip install streamlit pyngrok

In [120]:
from pyngrok import ngrok

ngrok.set_auth_token("33jiJarT6VDur40IP8SI65Iqh1z_7LTg1FGb7iLdYpnynERWg")

In [123]:
!streamlit run app.py &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.221.182.123:8501

  Stopping...


In [124]:
!pkill -f streamlit

In [125]:
!streamlit run app.py &>/dev/null &

In [127]:
from pyngrok import ngrok
import time

# 🔐 Set your ngrok auth token
ngrok.set_auth_token("33jiJarT6VDur40IP8SI65Iqh1z_7LTg1FGb7iLdYpnynERWg")

# 🚀 Run Streamlit in background
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

# ⏳ Thoda wait karo (important)
time.sleep(5)

# 🌐 Connect ngrok
public_url = ngrok.connect(8501)
print("Your App URL:", public_url)


Your App URL: NgrokTunnel: "https://superspiritually-unproven-phebe.ngrok-free.dev" -> "http://localhost:8501"
